In [1]:
import pandas as pd
import numpy as np
import math
import networkx as nx

from collections import defaultdict
from sklearn.preprocessing import normalize
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import pairwise_distances, precision_score, recall_score, f1_score, accuracy_score

from joblib import Parallel, delayed

In [2]:
def minkowski_distance(x, y, p):
    distance = sum(abs(xi - yi) ** p for xi, yi in zip(x, y)) ** (1 / p)
    return distance

def find_maximal_cliques(adj):
    # 创建图
    G = nx.Graph(adj)
    # 找到所有极大团
    cliques = list(nx.find_cliques(G))
    return cliques

def ig_find_maximal_cliques(adj):
    # 使用 igraph 库来高效地找到极大团
    G = ig.Graph.Adjacency((adj > 0).tolist())
    cliques = G.maximal_cliques()
    return cliques

def matching_degree(G, h_G, y):    
    phi_G_y = sum(h_G[x] for x in G if minkowski_distance(y, X_train[x], 2) < delta)
    return phi_G_y

def argmaxphi(phi_values, labels):
    total_phi = sum(phi_values)
    if total_phi == 0:
        return ''
    
    label_aggregated_phi = defaultdict(float)
    for i, value in enumerate(phi_values):
        label_aggregated_phi[labels[i]] += value / total_phi
        
    return max(label_aggregated_phi, key=label_aggregated_phi.get)

In [ ]:
data = pd.read_csv('banknote.csv')

y = data.iloc[:, -1].to_numpy()
scaler = MinMaxScaler()
X = scaler.fit_transform(data.iloc[:, :-1])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

n, m = X_train.shape

print('# of test:', len(y_test))

delta = 0.15 * math.sqrt(m)
alpha = 0.7

print('delta: ', delta, 'alpha: ', alpha)

dis_arr = np.zeros((n,n))
neighborhoods = []
neighborhoods_label = []
granules = []

for i in range(n):
    neighbors = []
    temp_label = []
    for j in range(n):
        dis_arr[i][j] = minkowski_distance(X_train[i], X_train[j], 2)
        if dis_arr[i][j] < delta:
            neighbors.append(j)
            temp_label.append(y_train[j])
    
    count = pd.Series(temp_label).value_counts()
    if not count[count > alpha * len(neighbors)].empty:
        neighborhoods.append(i)
        granules.append(neighbors)
        neighborhoods_label.append(count[count > alpha * len(neighbors)].index.tolist()[0])

print('Total: ', n, 'Filtered: ', len(neighborhoods))
acc = 0
dec = 0

for i, t in enumerate(X_test):
    mc_labels = []
    for j in neighborhoods:
        ob = X_train[j]
        if minkowski_distance(t, ob, 2) < delta:
            mc_labels.append(y_train[j])
    if mc_labels:
        dec += 1
        percentage = pd.Series(mc_labels).value_counts(normalize=True) * 100
        res = ', '.join([f"{element}: {pct:.2f}%" for element, pct in percentage.items()])
        if mc_labels[0]==y_test[i]:
            acc += 1
    else:
        res = ''
    # print(y_test[i], res)
print('qualit: ', acc/dec, dec/len(y_test))




acc = 0
dec = 0
h_Gs = []
for i, G in enumerate(granules):
    f_G ={}
    for x in G:
        count = sum(1 for k in G if minkowski_distance(X_train[k], X_train[x], 2) < delta)
        f_G[x] = count / len(G)
    total_f_G = sum(f_G.values())
    h_G = {x: f_G[x] / total_f_G for x in f_G}
    h_Gs.append(h_G)
    progress = (i+1) / len(granules) * 100
    #print(f"Progress: {progress:.2f}%", end='\r')
    
for i, t in enumerate(X_test):
    best_value = 0
    res = ''
    # for k, granule in enumerate(granules):
    #     phi_value = matching_degree(granule, h_Gs[k], t)
    #     if best_value < phi_value:
    #         best_value = phi_value
    #         res = neighborhoods_label[k]
    phi_values = [matching_degree(granule, h_Gs[k], t) for k, granule in enumerate(granules)]
    res = argmaxphi(phi_values, neighborhoods_label)
    if res:
        dec+=1
    if y_test[i]==res:
        acc+=1
    progress = (i+1) / len(X_test) * 100
    #print(f"Progress: {progress:.2f}%     ", end='\r')
    #print(y_test[i], res)

print(f"\nquanti: {(acc/dec*100):.2f}%", dec/len(y_test))

# of test: 275
delta:  0.3 alpha:  0.7
Total:  1097 Filtered:  885
0 0: 100.00%
0 0: 90.82%, 1: 9.18%
0 0: 92.00%, 1: 8.00%
0 0: 99.48%, 1: 0.52%
1 1: 100.00%
1 1: 100.00%
1 1: 90.62%, 0: 9.38%
0 0: 97.25%, 1: 2.75%
0 0: 100.00%
1 1: 100.00%
1 1: 98.18%, 0: 1.82%
1 1: 100.00%
1 1: 92.95%, 0: 7.05%
0 1: 61.27%, 0: 38.73%
1 1: 99.31%, 0: 0.69%
1 0: 59.05%, 1: 40.95%
1 1: 100.00%
1 1: 100.00%
1 1: 96.47%, 0: 3.53%
0 0: 100.00%
1 1: 100.00%
0 0: 100.00%
0 0: 93.57%, 1: 6.43%
0 1: 50.48%, 0: 49.52%
1 1: 100.00%
1 1: 88.24%, 0: 11.76%
1 1: 100.00%
1 1: 78.74%, 0: 21.26%
0 0: 100.00%
0 0: 100.00%
1 1: 100.00%
0 0: 76.80%, 1: 23.20%
1 1: 80.00%, 0: 20.00%
1 1: 99.12%, 0: 0.88%
1 1: 90.07%, 0: 9.93%
0 1: 83.08%, 0: 16.92%
0 0: 98.88%, 1: 1.12%
0 0: 55.72%, 1: 44.28%
0 0: 100.00%
0 0: 100.00%
1 1: 82.46%, 0: 17.54%
1 1: 99.30%, 0: 0.70%
1 1: 100.00%
0 0: 100.00%
0 0: 100.00%
1 1: 97.63%, 0: 2.37%
0 1: 52.99%, 0: 47.01%
0 0: 100.00%
0 0: 58.08%, 1: 41.92%
0 0: 76.72%, 1: 23.28%
0 1: 68.97%, 0: 31

In [15]:
import pandas as pd
import numpy as np
import math
import networkx as nx
from collections import defaultdict
from sklearn.preprocessing import normalize, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import pairwise_distances, precision_score, recall_score, f1_score, accuracy_score
from joblib import Parallel, delayed

def minkowski_distance(x, y, p):
    distance = sum(abs(xi - yi) ** p for xi, yi in zip(x, y)) ** (1 / p)
    return distance

def find_maximal_cliques(adj):
    # 创建图
    G = nx.Graph(adj)
    # 找到所有极大团
    cliques = list(nx.find_cliques(G))
    return cliques

def matching_degree(G, h_G, y, X_train, delta):
    # 计算匹配度
    phi_G_y = sum(h_G[x] for x in G if minkowski_distance(y, X_train[x], 2) < delta)
    return phi_G_y

def majority_vote(labels):
    """
    执行多数投票，返回出现次数最多的标签。
    如果存在多种可能的标签，选择第一个出现的标签。
    """
    if not labels:
        return None
    counts = pd.Series(labels).value_counts()
    return counts.idxmax()

# 读取并预处理数据
data = pd.read_csv('image.csv')
y = data.iloc[:, -1].to_numpy()
scaler = MinMaxScaler()
X = scaler.fit_transform(data.iloc[:, :-1])

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

n, m = X_train.shape

print('# of test samples:', len(y_test))



# 计算训练集之间的距离矩阵
neighborhoods = []
neighborhoods_label = []
granules = []

# 计算距离矩阵并筛选满足条件的邻域
beta = 2
alpha = 0.85

dis_arr = pairwise_distances(X_train, metric='euclidean')
distances = dis_arr[np.triu_indices_from(dis_arr, k=1)]
delta = np.percentile(distances, beta)

print('delta: ', delta, 'alpha: ', alpha)

# 并行处理每个训练样本
def process_training_sample(i):
    # 找到与样本 i 距离小于 delta 的所有样本索引
    neighbors = np.where(dis_arr[i] < delta)[0]
    temp_label = y_train[neighbors]
    count = pd.Series(temp_label).value_counts()
    if not count[count > alpha * len(neighbors)].empty:
        label = count[count > alpha * len(neighbors)].index.tolist()[0]
        return i, neighbors, label
    else:
        return None

# 使用多线程并行处理
results = Parallel(n_jobs=8)(delayed(process_training_sample)(i) for i in range(n))

# 过滤结果
for res in results:
    if res is not None:
        i, neighbors, label = res
        neighborhoods.append(i)
        granules.append(neighbors)
        neighborhoods_label.append(label)

# 将 neighborhoods 转换为 NumPy 数组以便于索引
neighborhoods = np.array(neighborhoods)

k = len(neighborhoods)

print('Total training samples:', n, 'Filtered neighborhoods:', len(neighborhoods))

# 定义用于计算每个granule的h_G的函数
def compute_h_G(G, X_train, delta):
    f_G = {}
    for x in G:
        # 使用列表生成式和numpy加速计数
        # 计算X_train[G]与X_train[x]的距离，并统计小于delta的个数
        distances = np.linalg.norm(X_train[G] - X_train[x], axis=1)
        count = np.sum(distances < delta)
        f_G[x] = count / len(G)
    total_f_G = sum(f_G.values())
    if total_f_G == 0:
        h_G = {x: 0 for x in f_G}
    else:
        h_G = {x: f_G[x] / total_f_G for x in f_G}
    return h_G

# 并行计算所有granules的h_Gs
h_Gs = Parallel(n_jobs=8, backend="loky")(
    delayed(compute_h_G)(G, X_train, delta) for G in granules
)

def weighted_vote(labels, weights):
    """
    执行加权投票，返回权重总和最高的标签。
    如果存在多种可能的标签，选择权重总和最高的标签。
    """
    if not labels or not weights:
        return None
    label_weights = defaultdict(float)
    for label, weight in zip(labels, weights):
        label_weights[label] += weight
    # 找到权重总和最高的标签
    return max(label_weights, key=label_weights.get)

# 定义预测测试样本的函数，使用加权投票
def predict_sample(i, t):
    # 计算测试样本与所有选定的训练样本的距离
    distances = np.linalg.norm(X_train[neighborhoods] - t, axis=1)
    mask = distances < delta
    if np.any(mask):
        close_indices = np.array(neighborhoods)[mask]
        mc_labels = y_train[close_indices].tolist()
        # 计算匹配度
        phi_values = [matching_degree(granule, h_Gs[idx], t, X_train, delta) for idx, granule in enumerate(granules)]
        # 获取匹配度最高的k个granule的索引
        top_k_indices = np.argsort(phi_values)[-k:]
        # 获取这些granule对应的标签和对应的phi_values
        top_k_labels = [neighborhoods_label[idx] for idx in top_k_indices]
        top_k_weights = [phi_values[idx] for idx in top_k_indices]
        # 执行加权投票
        res = weighted_vote(top_k_labels, top_k_weights)
        return (y_test[i], res)
    else:
        return (y_test[i], None)

# 使用Joblib进行并行预测
test_results = Parallel(n_jobs=8)(
    delayed(predict_sample)(i, t)
    for i, t in enumerate(X_test)
)

# 收集预测结果和真实标签
y_true = []
y_pred = []
for true_label, pred_label in test_results:
    y_true.append(true_label)
    y_pred.append(pred_label)

# 过滤无法预测的样本
y_true_filtered = []
y_pred_filtered = []
for true_label, pred_label in zip(y_true, y_pred):
    if pred_label is not None:
        y_true_filtered.append(true_label)
        y_pred_filtered.append(pred_label)

# 计算评价指标
if y_pred_filtered:
    coverage = len(y_pred_filtered) / len(y_test)
    accuracy = accuracy_score(y_true_filtered, y_pred_filtered)
    precision = precision_score(y_true_filtered, y_pred_filtered, average='weighted', zero_division=0)
    recall = accuracy * coverage
    f1 = 2 * (precision * recall) / (precision + recall)
else:
    coverage = 0
    accuracy = 0
    precision = 0
    recall = 0
    f1 = 0

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Coverage: {coverage:.4f}")

# of test samples: 42
delta:  0.2806749576732663 alpha:  0.85
Total training samples: 168 Filtered neighborhoods: 133
Accuracy: 1.0000
Precision: 1.0000
Recall: 0.7857
F1 Score: 0.8800
Coverage: 0.7857
